### Imports

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))


from src.paths import *
from src.data_validation import *

### Read GIS inputs

In [2]:
inputs = {
    "grid_substations": (GIS / "grid_substations.csv", "grid_substations_template.csv"),
    "transmission_lines": (GIS / "transmission_lines.csv", "transmission_lines_template.csv"),
    "solar_plants": (GIS / "solar_power_plants.csv", "solar_plants_template.csv"),
}
missing = [(name, path, template) for name, (path, template) in inputs.items() if not path.is_file()]
inputs_available = not missing

if missing:
    print("Validation skipped: required source datasets are not present yet.")
    print(f"Place completed CSVs in {GIS}:")
    for name, path, template in missing:
        print(f"- {path.name} (start from datasets/templates/{template})")
    # Empty typed frames keep later notebook cells runnable without inventing data.
    substations = pd.DataFrame(columns=["bus_id", "name", "latitude", "longitude", "voltage_kv"])
    lines = pd.DataFrame(columns=["line_id", "from_bus", "to_bus", "length_km"])
    solar_plants = pd.DataFrame(columns=["plant_id", "name", "bus_id", "capacity_mw"])
else:
    substations = read_and_validate(inputs["grid_substations"][0], "grid_substations")
    lines = read_and_validate(inputs["transmission_lines"][0], "transmission_lines")
    solar_plants = read_and_validate(inputs["solar_plants"][0], "solar_plants")

    display(substations.head())
    display(lines.head())
    display(solar_plants.head())

,bus_id,name,latitude,longitude,voltage_kv
0,BIYAGAMA,Biyagama Grid Substation,6.945,79.995,220
1,KOTMALE,Kotmale Hydro Switchyard,7.065,80.598,220
2,NEW_ANURADHAPURA,New Anuradhapura Substation,8.325,80.425,220
3,HABARANA,Habarana Grid Substation,8.035,80.752,220
4,VEYANGODA,Veyangoda Grid Substation,7.155,80.055,220


,line_id,from_bus,to_bus,length_km
0,L01,KOTMALE,BIYAGAMA,70.0
1,L02,KOTMALE,NEW_ANURADHAPURA,145.0
2,L03,NEW_ANURADHAPURA,HABARANA,45.0
3,L04,HABARANA,VEYANGODA,130.0
4,L05,NOROCHCHOLAI,NEW_ANURADHAPURA,100.0


,plant_id,name,bus_id,latitude,longitude,capacity_mw
0,SOL_01,Siyambalanduwa Solar Park,MONARAGALA,6.875,81.345,100
1,SOL_02,Hambantota Solar Park,HAMBANTOTA,6.140,81.120,10
2,SOL_03,Buruthakanda Solar,HAMBANTOTA,6.175,81.150,10
3,SOL_04,Welikanda Solar,VALACHCHENAI,7.950,81.250,10
4,SOL_05,Vavunathivu Solar,AMPARA,7.680,81.650,10


### Clean data types

In [3]:
substations["bus_id"] = substations["bus_id"].astype(str)

for col in ["latitude", "longitude", "voltage_kv"]:

    substations[col] = pd.to_numeric(
        substations[col],
        errors="coerce"
    )

lines["from_bus"] = lines["from_bus"].astype(str)
lines["to_bus"] = lines["to_bus"].astype(str)

lines["length_km"] = pd.to_numeric(
    lines["length_km"],
    errors="coerce"
)

solar_plants["bus_id"] = solar_plants["bus_id"].astype(str)

solar_plants["capacity_mw"] = pd.to_numeric(
    solar_plants["capacity_mw"],
    errors="coerce"
)

substations = substations.dropna(
    subset=["bus_id", "latitude", "longitude", "voltage_kv"]
)

lines = lines.dropna(
    subset=["from_bus", "to_bus", "length_km"]
)

solar_plants = solar_plants.dropna(
    subset=["bus_id", "capacity_mw"]
)

### Validate connectivity references

In [4]:
validate_grid(substations, lines)

known_buses = set(substations["bus_id"])

connected_solar = solar_plants[
    solar_plants["bus_id"].isin(known_buses)
]

unconnected_solar = solar_plants[
    ~solar_plants["bus_id"].isin(known_buses)
]

print("Connected solar plants:", len(connected_solar))
print("Unconnected solar plants:", len(unconnected_solar))

Connected solar plants: 10
Unconnected solar plants: 0


### Data quality summary

In [5]:
quality = pd.DataFrame({
    "Dataset": [
        "Substations",
        "Transmission lines",
        "Solar plants",
        "Hourly system profile"
    ],

    "Records": [
        len(substations),
        len(lines),
        len(solar_plants),
        len(profile) if "profile" in globals() else "Not loaded"
    ]
})

display(quality)

,Dataset,Records
0,Substations,30
1,Transmission lines,39
2,Solar plants,10
3,Hourly system profile,Not loaded
